In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
from skimage.filters import frangi, threshold_otsu
from skimage.morphology import closing, opening, disk
from skimage.exposure import equalize_adapthist
from sklearn.metrics import jaccard_score, f1_score, accuracy_score, confusion_matrix

DATA_DIR = Path.cwd() / "data"
ORIGINAL_DIR = DATA_DIR / "original"
LABELS_DIR = DATA_DIR / "labels"

In [ ]:
def split_channels(image: np.ndarray):
    """Return R, G, B channels from RGB image."""
    return image[:, :, 0], image[:, :, 1], image[:, :, 2]

In [ ]:
def load_image(path: Path) -> np.ndarray:
    return np.array(Image.open(path))


def load_label(path: Path) -> np.ndarray:
    lbl = np.array(Image.open(path))
    return (lbl > 0).astype(np.uint8)

In [ ]:
def create_foreground_mask(image: np.ndarray, threshold: int = 30) -> np.ndarray:
    return np.sum(image, axis=2) > threshold


def apply_mask(data: np.ndarray, mask: np.ndarray) -> np.ndarray:
    """Zero out pixels outside mask."""
    return data * mask

In [ ]:
from skimage.filters import gaussian
def enhance_green_channel(
    green: np.ndarray,
    mask: np.ndarray,
    clip_limit: float = 0.03,
    nbins: int = 256,
) -> np.ndarray:
    """Apply CLAHE to green channel within foreground mask."""
    green_blurred = gaussian(green, sigma=2)

    normalized = green_blurred.astype(float) / 255.0
    normalized[~mask] = 0
    enhanced = equalize_adapthist(normalized, kernel_size=None, clip_limit=clip_limit, nbins=nbins)
    enhanced[~mask] = 0
    return enhanced

In [ ]:
def frangi_vesselness(
    image: np.ndarray,
    sigmas: tuple = (1, 2, 3),
    alpha: float = 0.5,
    beta: float = 0.5,
    gamma: float = 15.0,
    scale_step=2,
    black_ridges: bool = True,
) -> np.ndarray:
    """Apply Frangi vesselness filter. Returns vessel probability map."""
    return frangi(image, sigmas=sigmas, alpha=alpha, beta=beta, gamma=gamma, black_ridges=black_ridges, scale_step=scale_step)

In [ ]:
def extract_veins(
    image: np.ndarray,
    sigmas: tuple = (1, 2, 3),
    clip_limit: float = 0.03,
    alpha: float = 0.5,
    beta: float = 0.5,
    gamma: float = 15.0,
) -> np.ndarray:
    """Extract veins from eye image. Returns binary vein mask."""
    _, green, _ = split_channels(image)
    fg_mask = create_foreground_mask(image)

    enhanced = enhance_green_channel(green, fg_mask, clip_limit=clip_limit)

    vesselness = frangi_vesselness(enhanced, sigmas=sigmas, alpha=alpha, beta=beta, gamma=gamma)
    vesselness_masked = apply_mask(vesselness, fg_mask)

    thresh = threshold_otsu(vesselness_masked[fg_mask])
    binary = vesselness_masked > thresh

    binary = closing(binary, disk(2))
    binary = opening(binary, disk(1))

    return binary.astype(np.uint8)

In [ ]:
def evaluate(pred: np.ndarray, gold: np.ndarray) -> dict:
    """Compute evaluation metrics between prediction and gold standard."""
    p_flat = pred.ravel()
    g_flat = gold.ravel()

    return {
        "accuracy": accuracy_score(g_flat, p_flat),
        "f1": f1_score(g_flat, p_flat, zero_division=0),
        "iou": jaccard_score(g_flat, p_flat, zero_division=0),
        "tn": int(confusion_matrix(g_flat, p_flat).ravel()[0]),
        "fp": int(confusion_matrix(g_flat, p_flat).ravel()[1]),
        "fn": int(confusion_matrix(g_flat, p_flat).ravel()[2]),
        "tp": int(confusion_matrix(g_flat, p_flat).ravel()[3]),
    }

In [ ]:
def display_images(*images, titles=None, cmap="gray", figsize_per_image=(4, 4)):
    """Display 1..N images in a horizontal row."""
    n = len(images)
    if n == 0:
        print("No images to display.")
        return
    if titles is None:
        titles = [f"Image {i + 1}" for i in range(n)]
    elif len(titles) != n:
        raise ValueError(f"{n} images but {len(titles)} titles")

    fig, axes = plt.subplots(1, n, figsize=(figsize_per_image[0] * n, figsize_per_image[1]))
    if n == 1:
        axes = [axes]
    for ax, img, title in zip(axes, images, titles):
        ax.imshow(img, cmap=cmap)
        ax.set_title(title)
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
import cv2
import numpy as np
from skimage import morphology
from scipy import ndimage

def create_retinal_mask(
    image_rgb: np.ndarray,
    smooth_kernel_size: int = 7,
    erode_ratio: float = 0.02,
    dilation_ratio: float = 0.02,
    min_area_ratio: float = 0.5
) -> np.ndarray:
    """
    Create a clean binary mask for a retinal fundus image.
    
    Args:
        image_rgb: Input RGB image (H, W, 3)
        smooth_kernel_size: Size of Gaussian blur kernel (must be odd)
        erode_ratio: Percentage of image width to erode (cleans edges)
        dilation_ratio: Percentage of image width to dilate (fills gaps)
        min_area_ratio: Minimum area of mask relative to image area
    
    Returns:
        Binary mask (uint8) where 1 = valid retina, 0 = background
    """
    
    # 1. Convert to grayscale - use Red channel (best contrast for fundus boundary)
    if image_rgb.ndim == 3:
        gray = image_rgb[:, :, 0]  # Red channel is index 0 in RGB
    else:
        gray = image_rgb.copy()
    
    # 2. Apply Gaussian blur to smooth noise
    gray_blur = cv2.GaussianBlur(gray, (smooth_kernel_size, smooth_kernel_size), 0)
    
    # 3. Otsu thresholding - automatically finds the cutoff
    _, binary = cv2.threshold(gray_blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # 4. Find largest contour (the circular field of view)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    
    if not contours:
        # Fallback: if no contour, use the whole image
        return np.ones_like(binary, dtype=np.uint8) * 255
    
    # Get the largest contour by area
    largest_contour = max(contours, key=cv2.contourArea)
    
    # Create mask from largest contour
    mask = np.zeros_like(binary, dtype=np.uint8)
    cv2.drawContours(mask, [largest_contour], -1, 255, -1)
    
    # 5. Erode slightly to remove edge artifacts (dark corners)
    height, width = mask.shape
    erode_kernel_size = max(1, int(min(height, width) * erode_ratio))
    if erode_kernel_size % 2 == 0:
        erode_kernel_size += 1
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (erode_kernel_size, erode_kernel_size))
    mask_eroded = cv2.erode(mask, kernel, iterations=1)
    
    # 6. Dilate slightly to fill small holes
    dilate_kernel_size = max(1, int(min(height, width) * dilation_ratio))
    if dilate_kernel_size % 2 == 0:
        dilate_kernel_size += 1
    
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (dilate_kernel_size, dilate_kernel_size))
    mask_final = cv2.dilate(mask_eroded, kernel, iterations=1)
    
    # 7. (Optional) Fill holes inside the mask
    mask_final = ndimage.binary_fill_holes(mask_final).astype(np.uint8) * 255
    
    # 8. Check if the mask is reasonable (not too small)
    mask_area = np.sum(mask_final > 0)
    image_area = height * width
    if mask_area < image_area * min_area_ratio:
        # Failed to find proper mask, return the original binary
        return binary.astype(np.uint8) * 255
    
    return mask_final.astype(np.uint8)


def create_retinal_mask_convex_hull(image_rgb: np.ndarray) -> np.ndarray:
    """
    Advanced retinal mask using convex hull + morphological closing.
    Best for images with irregular dark corners.
    """
    
    # 1. Red channel + median blur (removes salt/pepper noise)
    red = image_rgb[:, :, 0]
    red_blur = cv2.medianBlur(red, 9)
    
    # 2. Adaptive thresholding (works better than Otsu for uneven lighting)
    # invert=True because we want background dark, field bright
    binary = cv2.adaptiveThreshold(
        red_blur, 255, 
        cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
        cv2.THRESH_BINARY_INV, 
        51,  # block size (must be odd)
        15   # constant subtracted from mean
    )
    
    # 3. Find largest contour
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return np.ones_like(binary, dtype=np.uint8) * 255
    
    largest_contour = max(contours, key=cv2.contourArea)
    
    # 4. Convex hull - fills in all concave "dents" in the edge
    hull = cv2.convexHull(largest_contour)
    
    # 5. Draw hull
    mask = np.zeros_like(binary, dtype=np.uint8)
    cv2.drawContours(mask, [hull], -1, 255, -1)
    
    # 6. Morphological closing (fills gaps near edge)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (20, 20))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    # 7. Erode slightly to clean edge artifacts
    mask = cv2.erode(mask, kernel, iterations=1)
    
    return mask

In [ ]:
import cv2
from skimage.morphology import remove_small_objects
from skimage.filters import gaussian, apply_hysteresis_threshold

def normalize_and_remove_bright(image: np.ndarray, percentile: float = 95) -> np.ndarray:
    """Clips pixel intensities above a certain percentile to remove bright fragments 
    and normalizes the remaining intensities to the range [0, 255].
    """
    threshold = np.percentile(image, percentile)
    clipped = np.minimum(image, threshold)
    
    img_min = clipped.min()
    img_max = clipped.max()
    
    normalized = (clipped - img_min) / (img_max - img_min + 1e-7)
    return (normalized * 255).astype(np.uint8)

def process_one(image_id: str, tests: list[dict]) -> dict:
    """Load, extract veins, evaluate for a single image."""
    img = load_image(ORIGINAL_DIR / f"{image_id}.ppm")
    gold = load_label(LABELS_DIR / f"{image_id}.vk.ppm")

    red, green, blue = split_channels(img)
    fg_mask = create_retinal_mask_convex_hull(img)
    fg_mask = fg_mask > 0

    enhanced = enhance_green_channel(green, fg_mask)
    result = []

    for t in tests:
        print("Trying", t)
        vesselness = frangi_vesselness(enhanced, sigmas=t.get("sigmas"), alpha=0.5, beta=0.5, gamma=1, scale_step=None, black_ridges=True)
        vesselness_masked = apply_mask(vesselness, fg_mask)


        norm_vessel = cv2.normalize(vesselness_masked, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
        hyst_mask = apply_hysteresis_threshold(norm_vessel, low=10, high=60)

        # Morphological cleaning
        binary4 = binarye = closing(hyst_mask, disk(2))
        binary5 = binarye = opening(binarye, disk(1))
        binary6 = binarye = remove_small_objects(binarye, min_size=30)
        pred = binarye.astype(np.uint8)
        result.append((t, pred))
    


    return {
        "image_id": image_id,
        "image": img,
        "gold": gold,
        "red": red,
        "green": green,
        "blue": blue,
        "enhanced": enhanced,
        "vesselness": vesselness,
        "masked": vesselness_masked,

        "result": result,
    }

In [ ]:
r = process_one("im0001", [
    {"sigmas": (0.1, 0.5)},
    # {"sigmas": (1, 20)},
    # {"sigmas": (1, 50)},
    # {"sigmas": (1, 100)},
    # {"sigmas": (2, 10)},
    # {"sigmas": (2, 20)},
    # {"sigmas": (2, 50)},
    # {"sigmas": (2, 100)},
    # {"sigmas": (3, 10)},
    # {"sigmas": (3, 20)},
    {"sigmas": (3, 50)},
    # {"sigmas": (3, 100)},
    # {"sigmas": (4, 10)},
    # {"sigmas": (4, 20)},
    # {"sigmas": (4, 50)},
    # {"sigmas": (4, 100)},
    # {"sigmas": (6, 10)},
    # {"sigmas": (6, 50)},
    # {"sigmas": (6, 100)},
    # {"sigmas": (10, 10)},
    # {"sigmas": (10, 50)},
    # {"sigmas": (10, 100)},
    # {"sigmas": (10, 200)},
])
display_images(r["image"], r["gold"], r["red"], r["green"], r["blue"])
display_images(r["enhanced"], r["vesselness"])

for par, img in r["result"]:
    display_images(r["gold"], img, titles=["Label", f"pred {par}"])


In [ ]:
image_ids = sorted([p.stem for p in ORIGINAL_DIR.glob("*.ppm")])
image_ids = list(image_ids)[:1]
print(f"Found {len(image_ids)} images: {image_ids[:5]}...")

In [ ]:
results = []
for img_id in image_ids:
    r = process_one(img_id)
    results.append(r)
    print(f"{img_id}: acc={r['accuracy']:.3f}  f1={r['f1']:.3f}  iou={r['iou']:.3f}")

In [ ]:
df = pd.DataFrame(results)
df[["image_id", "accuracy", "f1", "iou", "tp", "fp", "fn", "tn"]]

In [ ]:
print(f"Mean accuracy: {df['accuracy'].mean():.3f}")
print(f"Mean F1:       {df['f1'].mean():.3f}")
print(f"Mean IoU:      {df['iou'].mean():.3f}")

In [ ]:
best_idx = df["f1"].idxmax()
worst_idx = df["f1"].idxmin()

visualize(
    results[best_idx]["image"],
    results[best_idx]["gold"],
    results[best_idx]["pred"],
    results[best_idx]["vesselness"],
    results[best_idx]["enhanced"],
    title=f"Best: {results[best_idx]['image_id']}  F1={results[best_idx]['f1']:.3f}",
)

In [ ]:
visualize(
    results[worst_idx]["image"],
    results[worst_idx]["gold"],
    results[worst_idx]["pred"],
    results[worst_idx]["vesselness"],
    results[worst_idx]["enhanced"],
    title=f"Worst: {results[worst_idx]['image_id']}  F1={results[worst_idx]['f1']:.3f}",
)